# LRS-Net Training

Lightweight Remote-Sensing Super-Resolution Network — GPU training notebook for Colab/Kaggle.

Run top to bottom. Everything that needs a GPU happens here; model comparison, the params/PSNR Pareto plot, and the downstream-task evaluation happen back on CPU locally using the weights, `history.csv`, and figures this notebook writes to Drive.

**Expected data layout on Drive** (see README.md in this folder for dataset suggestions):
```
DATA_DIR/
  HR/train/*.png   HR/val/*.png
  LR/train/*.png   LR/val/*.png   # omit LR/ entirely to use synthetic bicubic degradation instead
```

## 0. Check GPU

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Get the code
Clones the repo so `lrsnet/` is importable. Re-run this cell after pushing local changes to pull the latest version.

In [ ]:
import os

REPO_URL = "https://github.com/Wiredu2020/Super-Resolution-Methods-For-Satellite-Image-Applications.git"
REPO_DIR = "/content/Super-Resolution-Methods-For-Satellite-Image-Applications"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

import sys
sys.path.append(os.path.join(REPO_DIR, "Colab"))

## 2. Mount Drive
Used for reading the dataset and writing weights/figures back out. On Kaggle, replace this cell with the Kaggle input/output paths instead.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/SatelliteSR/data"        # adjust to your Drive path
OUTPUT_DIR = "/content/drive/MyDrive/SatelliteSR/outputs/LRSNet"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Config

In [ ]:
SCALE = 2
HR_SIZE = 512
BATCH_SIZE = 8
EPOCHS = 30
NUM_FILTERS = 32
NUM_BLOCKS = 6

HR_TRAIN_DIR = os.path.join(DATA_DIR, "HR/train")
HR_VAL_DIR = os.path.join(DATA_DIR, "HR/val")
# Set these to None to fall back to synthetic bicubic degradation instead of real LR pairs.
LR_TRAIN_DIR = os.path.join(DATA_DIR, "LR/train")
LR_VAL_DIR = os.path.join(DATA_DIR, "LR/val")

## 4. Datasets

In [ ]:
from lrsnet import data, losses, metrics, utils, build_lrsnet

train_ds = data.make_dataset(
    HR_TRAIN_DIR, LR_TRAIN_DIR, scale=SCALE, hr_size=HR_SIZE,
    batch_size=BATCH_SIZE, shuffle=True, augment=True,
)
val_ds = data.make_dataset(
    HR_VAL_DIR, LR_VAL_DIR, scale=SCALE, hr_size=HR_SIZE,
    batch_size=BATCH_SIZE, shuffle=False, augment=False,
)

## 5. Build LRS-Net

In [ ]:
model = build_lrsnet(num_filters=NUM_FILTERS, num_blocks=NUM_BLOCKS, scale=SCALE)
model.summary()
print(metrics.count_params(model))

## 6. Compile

In [ ]:
lr_schedule = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
    boundaries=[5000], values=[1e-4, 5e-5]
)
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

model.compile(
    optimizer=optimizer,
    loss=losses.l1_ssim_loss(),
    metrics=[metrics.PSNR, metrics.ssim_metric],
)

## 7. Callbacks

In [ ]:
weights_dir = os.path.join(OUTPUT_DIR, "weights")
os.makedirs(weights_dir, exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(weights_dir, "lrsnet_best.weights.h5"),
        save_best_only=True, save_weights_only=True, monitor="val_PSNR", mode="max",
    ),
    tf.keras.callbacks.CSVLogger(os.path.join(OUTPUT_DIR, "history_live.csv")),
]

## 8. Train

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 9. Save training figures, history, and final weights

In [ ]:
figures_dir = os.path.join(OUTPUT_DIR, "figures")

utils.plot_training_history(history, os.path.join(figures_dir, "lrsnet_loss_psnr.png"))
utils.save_history_csv(history, os.path.join(OUTPUT_DIR, "history.csv"))
model.save_weights(os.path.join(weights_dir, "lrsnet_final.weights.h5"))

## 10. Qualitative check

In [ ]:
lr_batch, hr_batch = next(iter(val_ds))
utils.visualize_predictions(
    model, lr_batch.numpy(), hr_batch.numpy(),
    os.path.join(figures_dir, "lrsnet_val_predictions.png"),
)

## 11. Efficiency report

In [ ]:
import json

report = utils.model_report(model)
print(report)
with open(os.path.join(OUTPUT_DIR, "model_report.json"), "w") as f:
    json.dump(report, f, indent=2)

## Bring back to CPU

Copy `OUTPUT_DIR` (weights, `history.csv`, `figures/`, `model_report.json`) from Drive into `Models/LRSNet/` locally, matching the existing `Models/SRCNN/`, `Models/EDSR/` layout. From there: the full-test-set comparison table, the params/PSNR Pareto plot against SRCNN/EDSR/SRGAN, and the downstream-task evaluation are all CPU-side work.